# 🎬 CineData Analytics - Agente Inteligente Text-to-SQL

Este projeto consiste no desenvolvimento de um agente inteligente capaz de realizar consultas e análises exploratórias em tempo real sobre o catálogo de filmes da **CineData Analytics**.

O objetivo principal é democratizar o acesso aos dados da camada Gold, permitindo que usuários não técnicos façam perguntas em linguagem natural sem a necessidade de escrever consultas SQL.

---

### 🧰 Stack Técnica
* **Linguagem:** Python
* **LLM / Provider:** OpenRouter API (Modelos Gratuitos: Gemini 2.0 / Qwen 2.5 Coder)
* **Banco de Dados:** SQLite3 (`cinerocket.db`)
* **Processamento de Dados:** Pandas

---

### 📐 Estrutura do Data Lakehouse (Modelagem Dimensional)
O banco de dados é composto por 10 tabelas conectadas por chave primária/estrangeira (`sk_movie_id`, `sk_person_id`, etc.):
* **Dimensões e Fatos Principais:** `dim_movies`, `fact_movies_performance`, `dim_genres`, `dim_people`, `dim_companies`, `dim_reviews`, `movie_reviews`
* **Tabelas Ponte (Bridge):** `bridge_movie_genre`, `bridge_movie_person`, `bridge_movie_company`

---



### 🤖 Arquitetura do Agente e Escolha do Framework

Para a implementação do agente **Text-to-SQL**, adotamos a seguinte estratégia:

* **Framework de Agentes:** **Custom Agent em Python Nativo (SDK OpenAI + OpenRouter API)**
  * **Motivação:** A escolha de um agente customizado em Python nativo garante execução leve e controle preciso das requisições, evitando *overhead* de bibliotecas e otimizando o consumo da cota gratuita da API.
* **Mecanismo de Fallback Automático:**
  * Para contornar limites de taxa (*Rate Limits / HTTP 429*) e indisponibilidade momentânea de provedores gratuitos do OpenRouter, o agente percorre dinamicamente uma lista prioritária de LLMs (`nvidia/nemotron-3.5-lightning:free`, `dots-studio/dots3-note-preview:free`, `apodex/apodex-1.1-mini:free`, `qwen/qwen3.8-27b:free`).
* **Execução Segura:**
  * O agente possui *guardrails* no prompt do sistema para garantir que **apenas consultas de leitura (`SELECT`)** sejam geradas e executadas com segurança sobre a camada Gold (`cinerocket.db`).

---

## 📊 Execução dos Casos de Uso e Análises
colocarei prints aqui depois

Importação do banco

In [4]:
import sqlite3

# Conectar ao banco de dados (ajuste o nome se não tiver renomeado)
db_name = "cinerocket.db"
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Obter a lista de todas as tabelas no banco
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("--- TABELAS ENCONTRADAS NO BANCO ---")
for t in tables:
    print(f"- {t[0]}")

print("\n" + "="*50 + "\n")

# Mostrar a estrutura DDL (CREATE TABLE) de cada tabela
print("--- ESTRUTURA DAS TABELAS (SCHEMA) ---")
cursor.execute("SELECT sql FROM sqlite_master WHERE type='table';")
schemas = cursor.fetchall()

for schema in schemas:
    if schema[0]:
        print(schema[0])
        print("-" * 40)

--- TABELAS ENCONTRADAS NO BANCO ---
- alembic_version
- dim_companies
- dim_genres
- dim_movies
- dim_people
- bridge_movie_company
- bridge_movie_genre
- bridge_movie_person
- dim_reviews
- fact_movies_performance
- movie_reviews


--- ESTRUTURA DAS TABELAS (SCHEMA) ---
CREATE TABLE alembic_version (
	version_num VARCHAR(32) NOT NULL, 
	CONSTRAINT alembic_version_pkc PRIMARY KEY (version_num)
)
----------------------------------------
CREATE TABLE dim_companies (
	sk_company_id VARCHAR(64) NOT NULL, 
	nome_produtora VARCHAR(255) NOT NULL, 
	CONSTRAINT pk_dim_companies PRIMARY KEY (sk_company_id), 
	CONSTRAINT uq_dim_companies_nome_produtora UNIQUE (nome_produtora)
)
----------------------------------------
CREATE TABLE dim_genres (
	sk_genre_id VARCHAR(64) NOT NULL, 
	nome_genero VARCHAR(50) NOT NULL, 
	CONSTRAINT pk_dim_genres PRIMARY KEY (sk_genre_id), 
	CONSTRAINT uq_dim_genres_nome_genero UNIQUE (nome_genero)
)
----------------------------------------
CREATE TABLE dim_movies (
	s

Instalação de biblioteca do OPENAI

In [13]:
!pip install openai pandas -q

In [14]:
import os
import re
import sqlite3
from contextlib import closing
from typing import Optional

import pandas as pd
from openai import (
    OpenAI,
    RateLimitError,
    APIStatusError,
    APIConnectionError,
    APITimeoutError,
)


# Configuração

DB_PATH = "cinerocket.db"
MAX_LINHAS = 200

MODELOS_GRATUITOS: list[str] = [
    "apodex/apodex-1.1-mini:free",
    "dots-studio/dots-3-note-preview:free",
    "qwen/qwen3.8-27b:free",
    "nvidia/nemotron-3.5-lightning:free",
]  # conferir os IDs em https://openrouter.ai/models?q=free


def _obter_api_key() -> str:
    """Lê a chave de variável de ambiente ou dos Secrets do Colab (nunca no código)."""
    chave = os.environ.get("OPENROUTER_API_KEY")
    if chave:
        return chave
    from google.colab import userdata  # painel 🔑 do Colab
    return userdata.get("OPENROUTER_API_KEY")


client = OpenAI(
    base_url="https://openrouter.ai/api/v1",  # string pura, sem formatação markdown
    api_key=_obter_api_key(),
    max_retries=0,  # o fallback entre modelos já faz o papel de retry
    timeout=60,
)

DB_SCHEMA = """
Tabelas e Relacionamentos no SQLite (cinerocket.db):

- dim_movies (sk_movie_id, id_filme, titulo, data_lancamento, ano_lancamento, duracao_minutos, idioma_original, status_filme, sinopse)
- fact_movies_performance (sk_movie_id, orcamento_usd, receita_usd, lucro_usd, orcamento_brl, receita_brl, lucro_brl, popularidade, nota_tmdb, qtd_tmdb, nota_imdb, qtd_imdb)
- dim_genres (sk_genre_id, nome_genero)
- bridge_movie_genre (sk_movie_id, sk_genre_id)
- dim_people (sk_person_id, nome_pessoa, tipo_pessoa ['Ator', 'Diretor', 'Roteirista'])
- bridge_movie_person (sk_movie_id, sk_person_id)
- dim_companies (sk_company_id, nome_produtora)
- bridge_movie_company (sk_movie_id, sk_company_id)
- dim_reviews (sk_review_id, sk_movie_id, qtd_avaliacoes_usuarios, nota_media_usuarios)
- movie_reviews (id, sk_movie_review_id, sk_movie_id, name, rating, text, created_at)
"""

PROMPT_SISTEMA = f"""
Você é um especialista em converter linguagem natural em consultas SQL para SQLite.

Esquema do Banco de Dados:
{DB_SCHEMA}

Regras:
1. Responda APENAS com a instrução SQL, dentro de um bloco ```sql ... ```.
2. Gere somente SELECT (ou WITH ... SELECT). NUNCA use INSERT, UPDATE, DELETE, DROP etc.
3. Para valores em R$, use as colunas terminadas em '_brl'.
4. Use apenas tabelas e colunas do esquema acima.
"""


# Extração e validação do SQL

_RE_THINK = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)
# Aceita ```sql, ```sqlite, ``` puro e até bloco sem fechamento (resposta truncada)
_RE_BLOCO = re.compile(r"```[a-zA-Z]*\s*(.*?)(?:```|\Z)", re.DOTALL)
_RE_SQL_SOLTO = re.compile(r"\b((?:SELECT|WITH)\b.*)", re.DOTALL | re.IGNORECASE)
_RE_INICIO_VALIDO = re.compile(r"^\s*(SELECT|WITH)\b", re.IGNORECASE)
_RE_COMENTARIOS = re.compile(r"--[^\n]*|/\*.*?\*/", re.DOTALL)
_RE_PROIBIDAS = re.compile(
    r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|ATTACH|DETACH|PRAGMA|VACUUM|REINDEX)\b",
    re.IGNORECASE,
)


def extrair_sql(texto: str) -> Optional[str]:
    """Extrai a consulta SQL da resposta do LLM; retorna None se não achar."""
    texto = _RE_THINK.sub("", texto or "")

    for bloco in _RE_BLOCO.findall(texto):
        if _RE_INICIO_VALIDO.match(bloco):
            return bloco.strip()

    m = _RE_SQL_SOLTO.search(texto)  # resposta sem cercas markdown
    return m.group(1).strip() if m else None


def validar_sql(sql: str) -> str:
    """Camada 1 (barata): rejeita qualquer coisa que não seja um único SELECT."""
    limpo = _RE_COMENTARIOS.sub(" ", sql).strip().rstrip(";").strip()
    if ";" in limpo:
        raise ValueError("Mais de uma instrução SQL detectada.")
    if not _RE_INICIO_VALIDO.match(limpo):
        raise ValueError("Apenas consultas SELECT/WITH são permitidas.")
    if _RE_PROIBIDAS.search(limpo):
        raise ValueError("Palavra-chave proibida na consulta.")
    return limpo


# Execução segura no SQLite

_ACOES_PERMITIDAS = {
    sqlite3.SQLITE_SELECT,
    sqlite3.SQLITE_READ,
    sqlite3.SQLITE_FUNCTION,
    sqlite3.SQLITE_RECURSIVE,
}


def _autorizador(acao, *_):
    return sqlite3.SQLITE_OK if acao in _ACOES_PERMITIDAS else sqlite3.SQLITE_DENY


def executar_sql(sql: str) -> pd.DataFrame:
    """Camada 2 (a que realmente protege): conexão somente leitura + authorizer."""
    with closing(sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)) as conn:
        conn.execute("PRAGMA query_only = ON")
        conn.set_authorizer(_autorizador)
        return pd.read_sql_query(sql, conn).head(MAX_LINHAS)


# Agente

def gerar_sql(pergunta: str, modelo: str) -> str:
    resposta = client.chat.completions.create(
        model=modelo,
        messages=[
            {"role": "system", "content": PROMPT_SISTEMA},
            {"role": "user", "content": pergunta},
        ],
        temperature=0.0,
    )
    # Modelos free às vezes devolvem 200 com choices vazio/None
    if not resposta.choices or not resposta.choices[0].message.content:
        raise ValueError("Resposta vazia do modelo.")

    sql = extrair_sql(resposta.choices[0].message.content)
    if sql is None:
        raise ValueError("Nenhum SQL encontrado na resposta.")
    return validar_sql(sql)


def perguntar_agente_com_fallback(pergunta: str) -> pd.DataFrame:
    """Tenta cada modelo da lista até obter um SQL válido que execute sem erro."""
    erros: list[str] = []

    for modelo in MODELOS_GRATUITOS:
        print(f" Tentando com: {modelo}...")
        try:
            sql = gerar_sql(pergunta, modelo)
        except (RateLimitError, APITimeoutError, APIConnectionError) as e:
            erros.append(f"{modelo}: indisponível ({type(e).__name__})")
            continue
        except APIStatusError as e:  # 404 (modelo removido), 5xx etc.
            erros.append(f"{modelo}: HTTP {e.status_code}")
            continue
        except ValueError as e:  # sem SQL / SQL rejeitado pelo guardrail
            erros.append(f"{modelo}: {e}")
            continue

        print(f" SQL gerado ({modelo}):\n{sql}\n")
        try:
            return executar_sql(sql)
        except (sqlite3.Error, pd.errors.DatabaseError) as e:
            erros.append(f"{modelo}: erro ao executar SQL ({e})")

    raise RuntimeError("Todos os modelos falharam:\n- " + "\n- ".join(erros))

In [15]:
pergunta = "Quais são os top 10 filmes com maior receita em R$?"
resultado = perguntar_agente_com_fallback(pergunta)
resultado

 Tentando com: apodex/apodex-1.1-mini:free...
 SQL gerado (apodex/apodex-1.1-mini:free):
SELECT m.titulo, f.receita_brl
FROM dim_movies m
JOIN fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
ORDER BY f.receita_brl DESC
LIMIT 10



,titulo,receita_brl
0,Avatar: The Way Of Water,1.239014e+10
1,Avengers: Endgame,1.109472e+10
2,Spider-man: No Way Home,1.097778e+10
3,Avengers: Infinity War,7.190431e+09
4,Top Gun: Maverick,7.160805e+09
5,Barbie,6.856159e+09
6,The Super Mario Bros. Movie,6.838414e+09
7,The Lion King,6.227552e+09
8,Frozen Ii,6.094028e+09
9,Jurassic World: Fallen Kingdom,4.934823e+09


## 📊 Execução dos Casos de Uso e Análises

In [16]:
# Pergunta 1: Lucro médio por gênero (apenas filmes com receita informada)
# Executa a pergunta
p1 = "Qual é o lucro médio em R$ por gênero, considerando apenas filmes com receita informada?"
df_lucro = perguntar_agente_com_fallback(p1)

# Identifica a coluna numérica e formata em Reais R$ (se df_lucro for um DataFrame)
if isinstance(df_lucro, pd.DataFrame):
    coluna_valor = df_lucro.columns[1]  # Pega a coluna de lucro/média
    df_lucro[coluna_valor] = df_lucro[coluna_valor].apply(lambda x: f"R$ {x:,.2f}" if pd.notnull(x) else "R$ 0,00")

df_lucro

 Tentando com: apodex/apodex-1.1-mini:free...
 SQL gerado (apodex/apodex-1.1-mini:free):
SELECT 
    g.nome_genero,
    AVG(f.lucro_brl) AS lucro_medio_r$ 
FROM 
    dim_genres g
JOIN 
    bridge_movie_genre bg ON g.sk_genre_id = bg.sk_genre_id
JOIN 
    fact_movies_performance f ON bg.sk_movie_id = f.sk_movie_id
WHERE 
    f.receita_brl IS NOT NULL
GROUP BY 
    g.nome_genero



,nome_genero,lucro_medio_r$
0,Action,"R$ 343,223,021.87"
1,Adventure,"R$ 514,678,791.39"
2,Animation,"R$ 303,517,897.37"
3,Comedy,"R$ 143,589,472.55"
4,Crime,"R$ 93,988,989.27"
5,Documentary,"R$ 5,028,959.44"
6,Drama,"R$ 84,161,852.72"
7,Family,"R$ 324,648,758.33"
8,Fantasy,"R$ 335,448,446.35"
9,History,"R$ 146,826,836.61"


In [17]:
# Pergunta 2: Filmes com maior margem de lucro (com receita e orçamento informados)
p2 = "Quais são os 10 filmes com maior margem de lucro, entre os que possuem receita e orçamento informados?"
perguntar_agente_com_fallback(p2)

 Tentando com: apodex/apodex-1.1-mini:free...
 SQL gerado (apodex/apodex-1.1-mini:free):
SELECT 
    m.titulo,
    f.receita_brl,
    f.orcamento_brl,
    f.lucro_brl,
    (f.lucro_brl / f.receita_brl) AS margem_lucro
FROM fact_movies_performance f
JOIN dim_movies m ON f.sk_movie_id = m.sk_movie_id
WHERE f.receita_brl IS NOT NULL
  AND f.orcamento_brl IS NOT NULL
  AND f.receita_brl > 0
ORDER BY margem_lucro DESC
LIMIT 10



,titulo,receita_brl,orcamento_brl,lucro_brl,margem_lucro
0,"Dad, I'm Sorry",9.530376e+07,712.12,9.530305e+07,0.999993
1,Etlb,3.225800e+06,161.29,3.225639e+06,0.999950
2,Jailbait,2.377884e+07,1688.44,2.377715e+07,0.999929
3,Trivikrama,5.232800e+04,20.93,5.230707e+04,0.999600
4,The Good Neighbor,4.869496e+05,538.72,4.864109e+05,0.998894
5,New York Masalı,1.643600e+03,3.29,1.640310e+03,0.997998
6,Secret Superstar,4.359820e+08,908293.25,4.350737e+08,0.997917
7,Alive,2.003480e+03,5.01,1.998470e+03,0.997499
8,Bad Ben: The Mandela Effect,4.129070e+05,1126.11,4.117809e+05,0.997273
9,Bad Ben,3.558830e+05,970.59,3.549124e+05,0.997273


In [18]:
# Pergunta 3: Maior divergência entre nota TMDB e nota IMDb
# Pergunta refinada
p3 = """Quais são os 10 filmes com maior divergência entre a nota TMDB e a nota IMDb,
considerando apenas filmes com notas maiores que zero (nota_tmdb > 0 e nota_imdb > 0)?"""

perguntar_agente_com_fallback(p3)

 Tentando com: apodex/apodex-1.1-mini:free...
 SQL gerado (apodex/apodex-1.1-mini:free):
SELECT 
    dim_movies.titulo,
    fact_movies_performance.nota_tmdb,
    fact_movies_performance.nota_imdb,
    ABS(fact_movies_performance.nota_tmdb - fact_movies_performance.nota_imdb) AS divergencia
FROM fact_movies_performance
JOIN dim_movies 
    ON fact_movies_performance.sk_movie_id = dim_movies.sk_movie_id
WHERE fact_movies_performance.nota_tmdb > 0 
    AND fact_movies_performance.nota_imdb > 0
ORDER BY divergencia DESC
LIMIT 10



,titulo,nota_tmdb,nota_imdb,divergencia
0,Country Music: Live At The Ryman,10.0,0.600,9.400
1,Cold Little Bird,10.0,0.727,9.273
2,Alfredo,1.0,10.000,9.000
3,Blade And Termeh,10.0,1.202,8.798
4,The Farmer,0.5,9.000,8.500
5,Coming Out: A 50 Year History,0.8,9.300,8.500
6,"Behave, Kid",1.0,9.400,8.400
7,Fan Club,1.0,9.300,8.300
8,Afraid Of The Night,1.0,9.300,8.300
9,Sketchy Times With Lilly Singh,10.0,1.700,8.300


In [19]:
# Pergunta 4: Nota média IMDb por ano de lançamento
p4 = "Qual é a nota média do IMDb por ano de lançamento dos filmes?"
perguntar_agente_com_fallback(p4)

 Tentando com: apodex/apodex-1.1-mini:free...
 SQL gerado (apodex/apodex-1.1-mini:free):
SELECT 
    m.ano_lancamento,
    AVG(f.nota_imdb) AS nota_media_imdb
FROM dim_movies m
JOIN fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
GROUP BY m.ano_lancamento
ORDER BY m.ano_lancamento



,ano_lancamento,nota_media_imdb
0,2016,6.338046
1,2017,6.339170
2,2018,6.271762
3,2019,6.259224
4,2020,6.244017
5,2021,6.230171
6,2022,6.230386
7,2023,6.225421
8,2024,6.154534
9,2025,6.575000


In [20]:
# Pergunta 5: Diretores com maior nota média (mínimo de 5 filmes)
# Pergunta 5 Refinada
p5 = """Quais são os 10 diretores com maior nota média (nota_tmdb), considerando apenas diretores com no mínimo 5 filmes e ignorando notas nulas ou zeradas?"""

perguntar_agente_com_fallback(p5)

 Tentando com: apodex/apodex-1.1-mini:free...
 SQL gerado (apodex/apodex-1.1-mini:free):
WITH director_films AS (
    SELECT 
        p.sk_person_id,
        p.nome_pessoa,
        COUNT(DISTINCT m.sk_movie_id) AS qtd_filmes,
        AVG(f.nota_tmdb) AS media_nota_tmdb
    FROM dim_people p
    JOIN bridge_movie_person bpm ON p.sk_person_id = bpm.sk_person_id
    JOIN dim_movies m ON bpm.sk_movie_id = m.sk_movie_id
    JOIN fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
    WHERE p.tipo_pessoa = 'Diretor'
        AND f.nota_tmdb IS NOT NULL 
        AND f.nota_tmdb != 0
    GROUP BY p.sk_person_id, p.nome_pessoa
    HAVING COUNT(DISTINCT m.sk_movie_id) >= 5
)
SELECT 
    nome_pessoa AS diretor,
    qtd_filmes,
    media_nota_tmdb
FROM director_films
ORDER BY media_nota_tmdb DESC
LIMIT 10



,diretor,qtd_filmes,media_nota_tmdb
0,Chad Payne,9,8.944444
1,Park Jun-soo,5,8.910600
2,John D. Boswell,7,8.865143
3,Mark Fischbach,5,8.860000
4,Tony Grech-smith,8,8.853125
5,David Freid,6,8.800000
6,Malte Wirtz,5,8.620000
7,Shun'ichi Yoshizawa,6,8.550000
8,Charles Roxburgh,5,8.500000
9,Vesna Busic-chaer,5,8.400000


In [21]:
p6 = """Qual é a dupla de ator e diretor que mais trabalhou junta em filmes?
Traga o nome do ator, o nome do diretor e a quantidade de filmes juntos, limitando aos 5 primeiros."""

perguntar_agente_com_fallback(p6)

 Tentando com: apodex/apodex-1.1-mini:free...
 SQL gerado (apodex/apodex-1.1-mini:free):
SELECT 
    a.nome_pessoa AS ator,
    d.nome_pessoa AS diretor,
    COUNT(*) AS qtd_filmes
FROM bridge_movie_person bmp_actor
JOIN dim_people a ON bmp_actor.sk_person_id = a.sk_person_id
JOIN bridge_movie_person bmp_director ON bmp_actor.sk_movie_id = bmp_director.sk_movie_id
JOIN dim_people d ON bmp_director.sk_person_id = d.sk_person_id
WHERE a.tipo_pessoa = 'Ator'
  AND d.tipo_pessoa = 'Diretor'
GROUP BY a.sk_person_id, d.sk_person_id
ORDER BY qtd_filmes DESC
LIMIT 5



,ator,diretor,qtd_filmes
0,Joe Anoa'i,Kevin Dunn,37
1,Colby Lopez,Kevin Dunn,32
2,David Love,Chad Payne,31
3,Anton Pelizzari,Chad Payne,31
4,Cameron Nichols,Chad Payne,31


In [22]:
p7 = """Quais são as 10 produtoras (dim_companies) com o maior lucro total em R$ (lucro_brl),
considerando apenas lucros maiores que zero?"""

perguntar_agente_com_fallback(p7)

 Tentando com: apodex/apodex-1.1-mini:free...
 SQL gerado (apodex/apodex-1.1-mini:free):
SELECT 
    dc.nome_produtora,
    SUM(fmp.lucro_brl) AS total_lucro_brl
FROM dim_companies dc
JOIN bridge_movie_company bmc ON dc.sk_company_id = bmc.sk_company_id
JOIN fact_movies_performance fmp ON bmc.sk_movie_id = fmp.sk_movie_id
WHERE fmp.lucro_brl > 0
GROUP BY dc.nome_produtora
ORDER BY total_lucro_brl DESC
LIMIT 10



,nome_produtora,total_lucro_brl
0,Marvel Studios,6.653452e+10
1,Universal Pictures,6.524954e+10
2,Walt Disney Pictures,4.816907e+10
3,Columbia Pictures,4.377135e+10
4,Warner Bros. Pictures,4.036535e+10
5,Paramount,2.878508e+10
6,Pascal Pictures,2.537040e+10
7,20th Century Fox,2.360245e+10
8,Illumination,2.054321e+10
9,New Line Cinema,1.817998e+10


In [23]:
# Pergunta 8 Refinada
p8 = """Quais são os 10 filmes mais avaliados considerando a soma das avaliações do TMDB e do IMDb,
apenas para filmes com avaliações maiores que zero em ambos (qtd_tmdb > 0 e qtd_imdb > 0)?"""

perguntar_agente_com_fallback(p8)

 Tentando com: apodex/apodex-1.1-mini:free...
 SQL gerado (apodex/apodex-1.1-mini:free):
SELECT 
    dm.titulo,
    fmp.qtd_tmdb + fmp.qtd_imdb AS qtd_avaliacoes
FROM dim_movies dm
JOIN fact_movies_performance fmp ON dm.sk_movie_id = fmp.sk_movie_id
WHERE fmp.qtd_tmdb > 0 AND fmp.qtd_imdb > 0
ORDER BY qtd_avaliacoes DESC
LIMIT 10



,titulo,qtd_avaliacoes
0,Joker,1746460
1,Avengers: Endgame,1508007
2,Avengers: Infinity War,1434495
3,Deadpool,1299233
4,Parasite,1209327
5,Oppenheimer,1097721
6,Dune,1097462
7,Spider-man: No Way Home,1088912
8,Once Upon A Time… In Hollywood,989133
9,Captain America: Civil War,968763


## Criação de interface de chat

In [24]:
import ipywidgets as widgets
from IPython.display import display, HTML

# 1. Criação dos elementos visuais do Chat
titulo_chat = widgets.HTML(value="<h3>💬 CineData Analytics - Chat do Agente Text-to-SQL</h3>")

caixa_historico = widgets.Output(layout=widgets.Layout(
    border='1px solid #ccc',
    height='300px',
    overflow_y='auto',
    padding='10px',
    margin='0 0 10px 0'
))

texto_pergunta = widgets.Text(
    placeholder='Digite a sua pergunta sobre os filmes (ex: Quais os top 5 filmes de ação?)...',
    layout=widgets.Layout(width='80%')
)

botao_enviar = widgets.Button(
    description='Enviar',
    button_style='primary',
    icon='paper-plane',
    layout=widgets.Layout(width='18%')
)

# 2. Função de envio da mensagem
def ao_clicar_enviar(b):
    pergunta_usuario = texto_pergunta.value.strip()
    if not pergunta_usuario:
        return

    # Limpa a caixa de texto
    texto_pergunta.value = ''

    with caixa_historico:
        # Exibe a pergunta do utilizador
        print(f"👤 Você: {pergunta_usuario}\n")
        print("🤖 Agente processando...\n")

        # Chama a função do agente com fallback
        resultado = perguntar_agente_com_fallback(pergunta_usuario)

        # Exibe o resultado (DataFrame ou mensagem de erro)
        if isinstance(resultado, pd.DataFrame):
            display(resultado)
        else:
            print(resultado)

        print("-" * 60 + "\n")

# Vincula o clique do botão e o 'Enter' do teclado
botao_enviar.on_click(ao_clicar_enviar)
texto_pergunta.on_submit(ao_clicar_enviar)

# 3. Monta e exibe a interface no Colab
caixa_entrada = widgets.HBox([texto_pergunta, botao_enviar])
chat_interface = widgets.VBox([titulo_chat, caixa_historico, caixa_entrada])

display(chat_interface)